In [9]:
%load_ext autoreload
%autoreload 2

import pandas as pd 
import numpy as np
import warnings
import openpyxl
from src.config import DATA_RAW, DATA_PROCESSED
from src.data_loader import load_processed, save_processed
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
df = load_processed('netflix_titles_cleaned.csv')
print(df.shape)
df.head()

(8709, 14)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021,9
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021,9
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021,9
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021,9
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021,9


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8709 entries, 0 to 8708
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8709 non-null   str           
 1   type          8709 non-null   str           
 2   title         8709 non-null   str           
 3   director      8709 non-null   str           
 4   cast          8709 non-null   str           
 5   country       8709 non-null   str           
 6   date_added    8709 non-null   datetime64[us]
 7   release_year  8709 non-null   int64         
 8   rating        8705 non-null   str           
 9   duration      8706 non-null   str           
 10  listed_in     8709 non-null   str           
 11  description   8709 non-null   str           
 12  year_added    8709 non-null   int64         
 13  month_added   8709 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(10)
memory usage: 3.9 MB


In [20]:
yearly_content = df.groupby('year_added').agg(
    total_titles=('show_id', 'count'),
    total_movies=('type', lambda x: (x == 'Movie').sum()),
    total_tv_shows=('type', lambda x: (x == 'TV Show').sum()),
    percentage_movies=('type', lambda x: (x == 'Movie').sum() / len(x) * 100),
    percentage_tv_shows=('type', lambda x: (x == 'TV Show').sum() / len(x) * 100)
).sort_values(by='year_added', ascending=True).reset_index()

yearly_content

,year_added,total_titles,total_movies,total_tv_shows,percentage_movies,percentage_tv_shows
0,2008,2,1,1,50.00,50.00
1,2009,2,2,0,100.00,0.00
2,2010,1,1,0,100.00,0.00
3,2011,13,13,0,100.00,0.00
4,2012,3,3,0,100.00,0.00
5,2013,10,6,4,60.00,40.00
6,2014,23,19,4,82.61,17.39
7,2015,73,56,17,76.71,23.29
8,2016,418,253,165,60.53,39.47
9,2017,1164,839,325,72.08,27.92


---

In [25]:
top_15_genres = df['listed_in'].str.split(', ', expand=True).stack().value_counts().head(15)

display(top_15_genres)

International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1328
Documentaries                869
Action & Adventure           859
Independent Movies           756
TV Dramas                    739
Children & Family Movies     641
Romantic Movies              616
Thrillers                    577
TV Comedies                  550
Crime TV Shows               459
Kids' TV                     433
Docuseries                   380
Name: count, dtype: int64

In [ ]:
df_country = df.dropna(subset=['country']).copy()
df_country = df_country[df_country['country'] != 'Unknown']

df_country['country'] = df_country['country'].str.split(',')
df_country = df_country.explode('country')
df_country['country'] = df_country['country'].str.strip()
df_country = df_country.reset_index(drop=True)

#------------
top_10_countries = df_country['country'].value_counts().head(10).index

df_top10 = df_country[df_country['country'].isin(top_10_countries)]

country_counts = pd.crosstab(df_top10['country'], df_top10['type'])
country_counts['Total'] = country_counts.sum(axis=1)
country_counts = country_counts.sort_values(by='Total', ascending=False)

print("--- TOP 10 COUNTRIES (COUNTS) ---")
print(country_counts)

#------------
country_pct = pd.crosstab(df_top10['country'], df_top10['type'], normalize='index') * 100
country_pct = country_pct.loc[country_counts.index]

print("\n--- MOVIE VS TV SHOW SHARE (%) ---")
print(country_pct.round(2))

--- TOP 10 COUNTRIES (COUNTS) ---
type            Movie  TV Show  Total
country                              
United States    2752      891   3643
India             962       83   1045
United Kingdom    534      253    787
Canada            319      113    432
France            303       86    389
Japan             119      195    314
Spain             171       57    228
South Korea        61      165    226
Germany           182       43    225
Mexico            111       58    169

--- MOVIE VS TV SHOW SHARE (%) ---
type            Movie  TV Show
country                       
United States   75.54    24.46
India           92.06     7.94
United Kingdom  67.85    32.15
Canada          73.84    26.16
France          77.89    22.11
Japan           37.90    62.10
Spain           75.00    25.00
South Korea     26.99    73.01
Germany         80.89    19.11
Mexico          65.68    34.32


---

In [39]:
df[df['type'] == 'TV Show'].value_counts('duration')

duration
1 Season      1791
2 Seasons      384
3 Seasons      178
4 Seasons       89
5 Seasons       55
6 Seasons       30
7 Seasons       18
8 Seasons       13
9 Seasons        8
10 Seasons       5
13 Seasons       2
15 Seasons       2
12 Seasons       2
17 Seasons       1
Name: count, dtype: int64

In [ ]:
rating_counts = pd.crosstab(df['rating'], df['type'])

# Add Total column and sort descending
rating_counts['Total'] = rating_counts.sum(axis=1)
rating_counts = rating_counts.sort_values(by='Total', ascending=False)

print("--- RATING COUNTS BY TYPE ---")
print(rating_counts)

# 2. Percentage Distribution (Share within Movies vs Share within TV Shows)
rating_pct = pd.crosstab(df['rating'], df['type'], normalize='columns') * 100
rating_pct = rating_pct.loc[rating_counts.index]  # Keep sorted order

print("\n--- PERCENTAGE SHARE WITHIN TYPE (%) ---")
print(rating_pct.round(2))

--- RATING COUNTS BY TYPE ---
type      Movie  TV Show  Total
rating                         
TV-MA      2062     1121   3183
TV-14      1427      706   2133
TV-PG       540      298    838
R           797        2    799
PG-13       490        0    490
TV-Y7       139      191    330
TV-Y        131      169    300
PG          287        0    287
TV-G        126       86    212
NR           75        3     78
G            41        0     41
TV-Y7-FV      5        0      5
NC-17         3        0      3
UR            3        0      3
66 min        1        0      1
84 min        1        0      1
74 min        1        0      1

--- PERCENTAGE SHARE WITHIN TYPE (%) ---
type      Movie  TV Show
rating                  
TV-MA     33.64    43.52
TV-14     23.28    27.41
TV-PG      8.81    11.57
R         13.00     0.08
PG-13      7.99     0.00
TV-Y7      2.27     7.41
TV-Y       2.14     6.56
PG         4.68     0.00
TV-G       2.06     3.34
NR         1.22     0.12
G          0.67     

In [48]:
# 1. Create month columns
df['month_name'] = df['date_added'].dt.month_name()
df['month_num'] = df['date_added'].dt.month

# 2. Total additions by month (Sorted chronologically Jan -> Dec)
monthly_counts = (
    df.dropna(subset=['date_added'])
    .groupby(['month_num', 'month_name'], observed=False)
    .size()
    .reset_index(name='total_additions')
)

# Identify the top month overall
top_month = monthly_counts.loc[monthly_counts['total_additions'].idxmax()]
print(f"Most additions occur in: {top_month['month_name']} ({top_month['total_additions']} titles)\n")

print("--- MONTHLY ADDITIONS BREAKDOWN ---")
print(monthly_counts[['month_name', 'total_additions']].to_string(index=False))

# 3. Monthly additions split by Type (Movie vs TV Show)
monthly_type_split = pd.crosstab(
    df['date_added'].dt.month_name(), 
    df['type']
)

# Reorder months chronologically
months_order = ['January', 'February', 'March', 'April', 'May', 'June', 
                'July', 'August', 'September', 'October', 'November', 'December']
monthly_type_split = monthly_type_split.reindex(months_order)

print("\n--- MOVIE VS TV SHOW SPLIT BY MONTH ---")
display(monthly_type_split)

Most additions occur in: July (819 titles)

--- MONTHLY ADDITIONS BREAKDOWN ---
month_name  total_additions
   January              727
  February              557
     March              734
     April              759
       May              626
      June              724
      July              819
    August              749
 September              765
   October              755
  November              697
  December              797

--- MOVIE VS TV SHOW SPLIT BY MONTH ---


type,Movie,TV Show
date_added,,
January,546,181
February,382,175
March,529,205
April,550,209
May,439,187
June,492,232
July,565,254
August,519,230
September,519,246
